# Digital Divide – Cusco
## Raster Metadata & Validity Audit

This notebook inspects two raster datasets used to analyse the digital divide in the Cusco region:

| File | Description | EPSG |
|---|---|---|
| `VNL_cusco_2025.tif` | NASA Black Marble nighttime radiance (nW·cm⁻²·sr⁻¹) | 4326 |
| `kernel_cobmovil2019_50m.tif` | Mobile coverage kernel density (50 m grid) | 32719 |

For each raster we report: CRS, shape, band count, NoData value, data type, bounding box, pixel resolution, valid pixel count, and value range.

In [1]:
import numpy as np
import rasterio
from pathlib import Path

DATA_DIR = Path("data")
FILES = {
    "VNL_cusco_2025 (Nighttime Radiance)": DATA_DIR / "VNL_cusco_2025.tif",
    "kernel_cobmovil2019_50m (Mobile Coverage KDE)": DATA_DIR / "kernel_cobmovil2019_50m.tif",
}

## Helper – resolution in approximate kilometres

For geographic CRS (degrees) we use the standard approximations:
- 1° latitude ≈ 111.32 km (nearly constant)
- 1° longitude ≈ 111.32 × cos(lat_centre) km

For projected CRS the resolution is already in metres.

In [2]:
def res_to_km(src):
    """Return (res_x_km, res_y_km) for a raster, regardless of CRS units."""
    crs = src.crs
    res_x, res_y = src.res          # always positive (rasterio convention)
    bounds = src.bounds

    if crs.is_geographic:           # units are degrees
        lat_centre = (bounds.top + bounds.bottom) / 2.0
        km_per_deg_lat = 111.32
        km_per_deg_lon = 111.32 * np.cos(np.radians(abs(lat_centre)))
        return res_x * km_per_deg_lon, res_y * km_per_deg_lat
    else:                           # units are metres (projected)
        return res_x / 1000.0, res_y / 1000.0

## Raster 1 – VNL_cusco_2025.tif
**NASA Black Marble nighttime radiance · EPSG:4326**

In [3]:
vnl_path = DATA_DIR / "VNL_cusco_2025.tif"

with rasterio.open(vnl_path) as src:
    crs      = src.crs
    height   = src.height
    width    = src.width
    bands    = src.count
    nodata   = src.nodata
    dtype    = src.dtypes[0]
    bounds   = src.bounds
    res_deg  = src.res                  # (x_deg, y_deg)
    res_km   = res_to_km(src)           # (x_km,  y_km)

    band = src.read(1)

# Valid pixels: no NoData declared, so treat every pixel as valid
# (the dataset may contain negative fill values; we flag them separately)
total_pixels = height * width
if nodata is not None:
    valid_mask = band != nodata
else:
    valid_mask = np.ones_like(band, dtype=bool)

valid_data   = band[valid_mask]
n_valid      = valid_mask.sum()
val_min      = float(valid_data.min())
val_max      = float(valid_data.max())

print("=" * 60)
print("VNL_cusco_2025.tif  –  Nighttime Radiance (nW·cm⁻²·sr⁻¹)")
print("=" * 60)
print(f"  CRS            : {crs}")
print(f"  Shape          : {height} rows × {width} cols  ({total_pixels:,} pixels total)")
print(f"  Band count     : {bands}")
print(f"  NoData value   : {nodata}")
print(f"  Data type      : {dtype}")
print(f"  Bounding box   :")
print(f"    West  : {bounds.left:.6f}°")
print(f"    East  : {bounds.right:.6f}°")
print(f"    South : {bounds.bottom:.6f}°")
print(f"    North : {bounds.top:.6f}°")
print(f"  Pixel resolution:")
print(f"    {res_deg[0]:.7f}° × {res_deg[1]:.7f}°  "
      f"(≈ {res_km[0]:.3f} km × {res_km[1]:.3f} km at scene centre)")
print(f"  Valid pixels   : {n_valid:,}  ({100*n_valid/total_pixels:.1f}% of total)")
print(f"  Value range    : {val_min:.4f}  →  {val_max:.4f} nW·cm⁻²·sr⁻¹")

VNL_cusco_2025.tif  –  Nighttime Radiance (nW·cm⁻²·sr⁻¹)
  CRS            : EPSG:4326
  Shape          : 1081 rows × 961 cols  (1,038,841 pixels total)
  Band count     : 1
  NoData value   : None
  Data type      : float32
  Bounding box   :
    West  : -74.002082°
    East  : -69.997916°
    South : -15.502084°
    North : -10.997917°
  Pixel resolution:
    0.0041667° × 0.0041667°  (≈ 0.451 km × 0.464 km at scene centre)
  Valid pixels   : 1,038,841  (100.0% of total)
  Value range    : -1.5000  →  1254.6145 nW·cm⁻²·sr⁻¹


## Raster 2 – kernel_cobmovil2019_50m.tif
**Mobile coverage kernel density · EPSG:32719 (UTM zone 19S)**

In [4]:
kde_path = DATA_DIR / "kernel_cobmovil2019_50m.tif"

with rasterio.open(kde_path) as src:
    crs2     = src.crs
    height2  = src.height
    width2   = src.width
    bands2   = src.count
    nodata2  = src.nodata
    dtype2   = src.dtypes[0]
    bounds2  = src.bounds
    res_m    = src.res                  # (x_m, y_m) – projected, metres
    res_km2  = res_to_km(src)           # convert to km

    band2 = src.read(1)

total_pixels2 = height2 * width2

# NoData is a very large negative float32 sentinel; use np.isclose for safety
if nodata2 is not None:
    valid_mask2 = ~np.isclose(band2, nodata2)
else:
    valid_mask2 = np.ones_like(band2, dtype=bool)

valid_data2 = band2[valid_mask2]
n_valid2    = int(valid_mask2.sum())
val_min2    = float(valid_data2.min())
val_max2    = float(valid_data2.max())

print("=" * 60)
print("kernel_cobmovil2019_50m.tif  –  Mobile Coverage KDE")
print("=" * 60)
print(f"  CRS            : {crs2}")
print(f"  Shape          : {height2} rows × {width2} cols  ({total_pixels2:,} pixels total)")
print(f"  Band count     : {bands2}")
print(f"  NoData value   : {nodata2:.6e}" if nodata2 is not None else "  NoData value   : None")
print(f"  Data type      : {dtype2}")
print(f"  Bounding box   (metres, UTM 19S):")
print(f"    West  (Easting) : {bounds2.left:.3f} m")
print(f"    East  (Easting) : {bounds2.right:.3f} m")
print(f"    South (Northing): {bounds2.bottom:.3f} m")
print(f"    North (Northing): {bounds2.top:.3f} m")
print(f"  Pixel resolution:")
print(f"    {res_m[0]:.1f} m × {res_m[1]:.1f} m  (= {res_km2[0]*1000:.1f} m × {res_km2[1]*1000:.1f} m)")
print(f"  Valid pixels   : {n_valid2:,}  ({100*n_valid2/total_pixels2:.1f}% of total)")
print(f"  Value range    : {val_min2:.6e}  →  {val_max2:.6e}")

kernel_cobmovil2019_50m.tif  –  Mobile Coverage KDE
  CRS            : EPSG:32719
  Shape          : 6116 rows × 7754 cols  (47,423,464 pixels total)
  Band count     : 1
  NoData value   : -3.402823e+38
  Data type      : float32
  Bounding box   (metres, UTM 19S):
    West  (Easting) : -43080.111 m
    East  (Easting) : 344619.889 m
    South (Northing): 8337100.059 m
    North (Northing): 8642900.059 m
  Pixel resolution:
    50.0 m × 50.0 m  (= 50.0 m × 50.0 m)
  Valid pixels   : 47,423,464  (100.0% of total)
  Value range    : 0.000000e+00  →  3.134648e-06


## Summary table

In [5]:
import pandas as pd

summary = pd.DataFrame({
    "Attribute": [
        "CRS",
        "Shape (H × W)",
        "Total pixels",
        "Band count",
        "NoData value",
        "Data type",
        "Bbox – West",
        "Bbox – East",
        "Bbox – South",
        "Bbox – North",
        "Pixel res (native units)",
        "Pixel res (≈ km)",
        "Valid pixels",
        "Valid pixels (%)",
        "Value min",
        "Value max",
    ],
    "VNL_cusco_2025": [
        "EPSG:4326 (geographic)",
        f"{height} × {width}",
        f"{height*width:,}",
        bands,
        str(nodata),
        dtype,
        f"{bounds.left:.6f}°",
        f"{bounds.right:.6f}°",
        f"{bounds.bottom:.6f}°",
        f"{bounds.top:.6f}°",
        f"{res_deg[0]:.7f}° × {res_deg[1]:.7f}°",
        f"{res_km[0]:.3f} × {res_km[1]:.3f}",
        f"{n_valid:,}",
        f"{100*n_valid/(height*width):.1f}%",
        f"{val_min:.4f}",
        f"{val_max:.4f}",
    ],
    "kernel_cobmovil2019_50m": [
        "EPSG:32719 (UTM 19S)",
        f"{height2} × {width2}",
        f"{height2*width2:,}",
        bands2,
        f"{nodata2:.3e}",
        dtype2,
        f"{bounds2.left:.1f} m",
        f"{bounds2.right:.1f} m",
        f"{bounds2.bottom:.1f} m",
        f"{bounds2.top:.1f} m",
        f"{res_m[0]:.1f} m × {res_m[1]:.1f} m",
        f"{res_km2[0]:.4f} × {res_km2[1]:.4f}",
        f"{n_valid2:,}",
        f"{100*n_valid2/total_pixels2:.1f}%",
        f"{val_min2:.4e}",
        f"{val_max2:.4e}",
    ],
})

summary.set_index("Attribute", inplace=True)
summary

,VNL_cusco_2025,kernel_cobmovil2019_50m
Attribute,,
CRS,EPSG:4326 (geographic),EPSG:32719 (UTM 19S)
Shape (H × W),1081 × 961,6116 × 7754
Total pixels,"1,038,841","47,423,464"
Band count,1,1
NoData value,None,-3.403e+38
Data type,float32,float32
Bbox – West,-74.002082°,-43080.1 m
Bbox – East,-69.997916°,344619.9 m
Bbox – South,-15.502084°,8337100.1 m


---
## Reprojection & Grid Alignment

**Goal:** bring the connectivity (KDE) raster onto the same coordinate reference system and pixel grid as the nighttime-light (VNL) raster so that every array index maps to the same ground location.

**Pipeline:**
1. **Reproject** KDE from EPSG:32719 (UTM 19S, metres) → EPSG:4326 (geographic, degrees) using **bilinear** resampling, preserving the natural ~50 m pixel size expressed in degrees.
2. **Resample** the reprojected KDE to the **exact grid** of the VNL raster (same `transform`, same `height × width`) — also bilinear.
3. **Verify** that both arrays share identical dimensions before any further analysis.

In [5]:
from rasterio.warp import reproject, Resampling, calculate_default_transform

kde_path = DATA_DIR / "kernel_cobmovil2019_50m.tif"
vnl_path = DATA_DIR / "VNL_cusco_2025.tif"

### Step 1 — Reproject KDE: EPSG:32719 → EPSG:4326 (bilinear)

`calculate_default_transform` derives the optimal output transform and grid size that preserves the source pixel footprint (~50 m) after conversion to degrees.

In [6]:
dst_crs = rasterio.crs.CRS.from_epsg(4326)

with rasterio.open(kde_path) as src:
    kde_src_transform = src.transform
    kde_src_crs       = src.crs
    kde_nodata        = src.nodata

    dst_transform, dst_width, dst_height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    kde_4326 = np.full((dst_height, dst_width), np.nan, dtype=np.float32)

    reproject(
        source=rasterio.band(src, 1),
        destination=kde_4326,
        src_transform=kde_src_transform,
        src_crs=kde_src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=kde_nodata,
        dst_nodata=np.nan,
    )

kde_4326_transform = dst_transform
kde_4326_crs       = dst_crs

print(f"Reprojected KDE shape  : {kde_4326.shape[0]:,} rows × {kde_4326.shape[1]:,} cols")
print(f"CRS                    : {kde_4326_crs}")
print(f"Transform              :\n{kde_4326_transform}")
print(f"Non-NaN pixels         : {int(np.sum(~np.isnan(kde_4326))):,}")

Reprojected KDE shape  : 6,132 rows × 7,905 cols
CRS                    : EPSG:4326
Transform              :
| 0.00, 0.00,-74.05|
| 0.00,-0.00,-12.23|
| 0.00, 0.00, 1.00|
Non-NaN pixels         : 47,192,434


### Step 2 — Resample to the exact VNL grid (bilinear)

The reprojected KDE is already in EPSG:4326 but sits on its own ~50 m native grid (6,132 × 7,905). Here we resample it onto the VNL transform (0.0041667° ≈ 463 m pixels, 1,081 × 961) so both arrays are pixel-aligned.

In [7]:
with rasterio.open(vnl_path) as vnl:
    vnl_transform = vnl.transform
    vnl_crs       = vnl.crs
    vnl_height    = vnl.height
    vnl_width     = vnl.width
    vnl_array     = vnl.read(1)

kde_aligned = np.full((vnl_height, vnl_width), np.nan, dtype=np.float32)

reproject(
    source=kde_4326,
    destination=kde_aligned,
    src_transform=kde_4326_transform,
    src_crs=kde_4326_crs,
    dst_transform=vnl_transform,
    dst_crs=vnl_crs,
    resampling=Resampling.bilinear,
    src_nodata=np.nan,
    dst_nodata=np.nan,
)

print(f"KDE aligned shape  : {kde_aligned.shape[0]:,} rows × {kde_aligned.shape[1]:,} cols")
print(f"Transform          :\n{vnl_transform}")
print(f"Non-NaN pixels     : {int(np.sum(~np.isnan(kde_aligned))):,}")
print(f"Value range        : {float(np.nanmin(kde_aligned)):.4e}  →  {float(np.nanmax(kde_aligned)):.4e}")

KDE aligned shape  : 1,081 rows × 961 cols
Transform          :
| 0.00, 0.00,-74.00|
| 0.00,-0.00,-11.00|
| 0.00, 0.00, 1.00|
Non-NaN pixels     : 566,663
Value range        : 0.0000e+00  →  3.1157e-06


### Step 3 — Verify identical dimensions

In [9]:
print(f"VNL shape          : {vnl_array.shape}")
print(f"KDE aligned shape  : {kde_aligned.shape}")
print(f"Shared transform   :\n{vnl_transform}")

assert vnl_array.shape == kde_aligned.shape, (
    f"Shape mismatch: VNL {vnl_array.shape} vs KDE {kde_aligned.shape}"
)

print("\nBoth arrays share identical shape and transform. Ready for analysis.")

VNL shape          : (1081, 961)
KDE aligned shape  : (1081, 961)
Shared transform   :
| 0.00, 0.00,-74.00|
| 0.00,-0.00,-11.00|
| 0.00, 0.00, 1.00|

Both arrays share identical shape and transform. Ready for analysis.


---
## Percentile-Based Normalization [0, 1]

**Procedure (applied identically to both layers):**
1. Replace negative values and NoData sentinels (`NaN`) with **0** — floors the distribution at zero before any clipping.
2. Compute the **2nd and 98th percentiles** on the full resulting array (including replaced zeros).
3. **Clip** values to `[p2, p98]` — suppresses extreme outliers at both tails.
4. **Scale** linearly to `[0, 1]`: `(x − p2) / (p98 − p2)`.

### VNL — Nighttime Radiance

In [8]:
vnl_norm = vnl_array.astype(np.float32).copy()

# Step 1 – floor at 0 (no declared NoData, but negatives are fill/noise)
vnl_norm[vnl_norm < 0] = 0.0

# Step 2 – percentile thresholds
p2_vnl  = float(np.percentile(vnl_norm, 2))
p98_vnl = float(np.percentile(vnl_norm, 98))

# Steps 3 & 4 – clip then scale
vnl_norm = np.clip(vnl_norm, p2_vnl, p98_vnl)
vnl_norm = (vnl_norm - p2_vnl) / (p98_vnl - p2_vnl)

print(f"VNL  p2 = {p2_vnl:.6f}   p98 = {p98_vnl:.6f}")
print(f"     min  = {vnl_norm.min():.4f}")
print(f"     max  = {vnl_norm.max():.4f}")
print(f"     mean = {vnl_norm.mean():.4f}")
print(f"     std  = {vnl_norm.std():.4f}")

VNL  p2 = 0.000000   p98 = 0.633384
     min  = 0.0000
     max  = 1.0000
     mean = 0.0391
     std  = 0.1767


### KDE — Mobile Coverage (aligned to VNL grid)

In [9]:
kde_norm = kde_aligned.copy()

# Step 1 – replace NaN (out-of-extent pixels) and any negatives with 0
kde_norm = np.where(np.isnan(kde_norm), 0.0, kde_norm)
kde_norm[kde_norm < 0] = 0.0

# Step 2 – percentile thresholds
p2_kde  = float(np.percentile(kde_norm, 2))
p98_kde = float(np.percentile(kde_norm, 98))

# Steps 3 & 4 – clip then scale (guard against degenerate range)
kde_norm = np.clip(kde_norm, p2_kde, p98_kde)
if p98_kde > p2_kde:
    kde_norm = (kde_norm - p2_kde) / (p98_kde - p2_kde)
else:
    kde_norm = np.zeros_like(kde_norm, dtype=np.float32)

print(f"KDE  p2 = {p2_kde:.6e}   p98 = {p98_kde:.6e}")
print(f"     min  = {kde_norm.min():.4f}")
print(f"     max  = {kde_norm.max():.4f}")
print(f"     mean = {kde_norm.mean():.4f}")
print(f"     std  = {kde_norm.std():.4f}")

KDE  p2 = 0.000000e+00   p98 = 3.262267e-07
     min  = 0.0000
     max  = 1.0000
     mean = 0.0389
     std  = 0.1688
